# TextSplitter的源码剖析

主要讲一下内部的三个方法（没有本质的区别，主要看参数类型和返回类型）：

方法1：
split_text(self, text: str) -> list[str]:
> 传入的参数类型：文本内容（或字符串），返回值类型：字符串列表
>
> 此方法是抽象方法，具体的实现细节由子类来决定

方法2：
create_documents(self, texts: list[str],...) -> list[Document]:
> 传入的参数类型：字符串列表，返回值类型：Document对象列表
>
> 此方法的底层调用了**split_text()**，即将参数中的每一个字符串都传入split_text()中执行，得到的字符串列表中，将字符串**封装**为Document对象，就构成了list[Document]。


方法3：
split_documents(self, documents: Iterable[Document]) -> list[Document]:
> 传入的参数类型：Document对象列表，返回值类型：Document对象列表
>
> 此方法的底层调用了**create_documents()**,将参数中的每一个Document对象，提取其page_content字段，则构成了字符串列表，然后调用方法2即可。

# 具体文档切分器的使用


## 1、CharacterTextSplitter：Split by character

举例1：字符串文本的分割

In [ ]:
from langchain_text_splitters import CharacterTextSplitter

text = """
Langchain是一个用于开发由语言模型驱动的应用程序的框架的。它提供了一套工具和抽象，让开发者能够更容易地构建复杂地应用程序。
"""

# 定义字符分割器
splitter = CharacterTextSplitter(
    chunk_size = 50,   # 每块大小，默认值为4000
    chunk_overlap = 5,    # 块与块之间地重复字符数
    # length_function = len,
    separator = ""   # 设置为空字符串时，表示禁用分隔符优先
)

# 分割文本
texts = splitter.split_text(text)

for i, chunk in enumerate(texts):
    print(f"块{i + 1}:长度:{len(chunk)}")
    print(chunk)
    print("-" * 50)

举例2：指定分割符

In [ ]:
# 1.导入相关依赖
from langchain_text_splitters import CharacterTextSplitter

# 2.定义要分割的文本
text = "这是一个示例文本啊。我们将使用CharacterTextSplitter将其分割成小块。分割基于字符数。"

# text = """
# LangChain 是一个用于开发由语言模型。驱动的应用程序的框架的。它提供了一套工具和抽象。使开发者能够更容易地构建复杂的应用程序。
# """

# 3.定义分割器实例
text_splitter = CharacterTextSplitter(
    chunk_size=30,   # 每个块的最大字符数
    chunk_overlap=5, # 块之间的重叠字符数
    separator="。",  # 按句号分割优先（注意）
)

# 4.开始分割
chunks = text_splitter.split_text(text)

# 5.打印效果
for  i,chunk in enumerate(chunks):
    print(f"块 {i + 1}:长度：{len(chunk)}")
    print(chunk)
    print("-"*50)


**separator优先原则**：当设置了 `separator`（如"。"），分割器会首先尝试在分隔符处分割，然后再考虑 chunk_size。这是为了避免在句子中间硬性切断。这种设计是为了：

1. 优先保持语义完整性（不切断句子）
2. 避免产生无意义的碎片（如半个单词/不完整句子）
3. 如果 `chunk_size` 比片段小，无法拆分片段，导致 overlap失效。
4. chunk_overlap仅在合并后的片段之间生效（如果 `chunk_size` 足够大）。如果没有合并的片段，则 overlap失效。

举例3：指定分隔符

In [ ]:
# 1.导入相关依赖
from langchain_text_splitters import CharacterTextSplitter

# 2.定义要分割的文本
text = "这是第一段文本。这是第二段内容。最后一段结束。"

# 3.定义字符分割器
text_splitter = CharacterTextSplitter(
    chunk_size=20,
    chunk_overlap=8,
    separator="。",
    # keep_separator=True #chunk中是否保留切割符
)

# 4.分割文本
chunks = text_splitter.split_text(text)

# 5.打印结果
for  i,chunk in enumerate(chunks):
    print(f"块 {i + 1}:长度：{len(chunk)}")
    print(chunk)
    print("-"*50)

## 2、RecursiveCharacterTextSplitter：最常用

举例1：使用split_text()方法演示

In [ ]:
# 1.导入相关依赖
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 2.定义RecursiveCharacterTextSplitter分割器对象
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=10,
    chunk_overlap=0,
    add_start_index=True,
)

# 3.定义拆分的内容
text="LangChain框架特性\n\n多模型集成(GPT/Claude)\n记忆管理功能\n链式调用设计。文档分析场景示例：需要处理PDF/Word等格式。"

# 4.拆分器分割
paragraphs = text_splitter.split_text(text)

for i,chunk in enumerate(paragraphs):
    print(f"块{i + 1},长度：{len(chunk)}")
    print(chunk)
    print('-' * 50)

举例2：使用create_documents()方法演示，传入字符串列表，返回Document对象列表

In [ ]:
# 1.导入相关依赖
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 2.定义RecursiveCharacterTextSplitter分割器对象
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=10,
    chunk_overlap=0,
    add_start_index=True,
)

# 3.定义分割的内容
# text="LangChain框架特性\n\n多模型集成(GPT/Claude)\n记忆管理功能\n链式调用设计。文档分析场景示例：需要处理PDF/Word等格式。"

list=["LangChain框架特性\n\n多模型集成(GPT/Claude)\n记忆管理功能\n链式调用设计。文档分析场景示例：需要处理PDF/Word等格式。"]

# 4.分割器分割
# create_documents()：形参是字符串列表，返回值是Document的列表
paragraphs = text_splitter.create_documents(list)


for para in paragraphs:
    print(para)
    print('-------')

举例3：使用create_documents()方法演示，将本地文件内容加载成字符串，进行拆分

In [ ]:
# 1.导入相关依赖
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 2.打开.txt文件
with open("../asset/load/09-ai.txt", encoding="utf-8") as f:
    state_of_the_union = f.read()  #返回的是字符串

# 3.定义RecursiveCharacterTextSplitter（递归字符分割器）
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=20,
    #chunk_overlap=0,
    length_function=len
)

# 4.分割文本
texts = text_splitter.create_documents([state_of_the_union])

# 5.打印分割文本
for text in texts:
    print(f"🔥{text.page_content}")

举例4：使用split_documents()方法演示，利用PDFLoader加载文档，对文档的内容用递归切割器切割

In [ ]:
# 1.导入相关依赖
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 2.定义PyPDFLoader加载器
loader = PyPDFLoader("../asset/load/04-load.pdf")

# 3.加载和切割文档对象
docs = loader.load()   # 返回Document对象构成的list
# print(f"第0页：\n{docs[0]}")

# 4.定义切割器
text_splitter = RecursiveCharacterTextSplitter(
    # chunk_size=200,
    chunk_size=120,
    chunk_overlap=0,
    # chunk_overlap=100,
    length_function=len,
    add_start_index=True,
)

# 5.对pdf内容进行切割得到文档对象
paragraphs = text_splitter.split_documents(docs)

for para in paragraphs:
    print(para)
    print('-------')

## 3、TokenTextSplitter/CharacterTextSplitter：Split by tokens

举例1：使用TokenTextSplitter

In [ ]:
# 1.导入相关依赖
from langchain_text_splitters import TokenTextSplitter

# 2.初始化 TokenTextSplitter
text_splitter = TokenTextSplitter(
    chunk_size=33,  # 最大 token 数为 33
    chunk_overlap=0, # 重叠 token 数为 0
    # model_name="gpt-4", # 选择 GPT-4 模型的编码器
    encoding_name="cl100k_base",  # 使用 OpenAI 的编码器,将文本转换为 token 序列
)
# 3.定义文本
text = "人工智能是一个强大的开发框架。它支持多种语言模型和工具链。人工智能是指通过计算机程序模拟人类智能的一门科学。自20世纪50年代诞生以来，人工智能经历了多次起伏。"

# 4.开始切割
texts = text_splitter.split_text(text)

# 打印分割结果
print(f"原始文本被分割成了 {len(texts)} 个块:")
for i, chunk in enumerate(texts):
    print(f"块 {i+1}: 长度：{len(chunk)} 内容：{chunk}")
    print("-" * 50)

举例2：使用CharacterTextSplitter

In [ ]:
# 1.导入相关依赖
from langchain_text_splitters import CharacterTextSplitter
import tiktoken  # 用于计算Token数量


# 2.定义通过Token切割器
text_splitter = CharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base", # 使用 OpenAI 的编码器
    chunk_size=18,
    chunk_overlap=0,
    separator="。",  # 指定中文句号为分隔符
    keep_separator=False,  # chunk中是否保留分隔符
)
# 3.定义文本
text = "人工智能是一个强大的开发框架。它支持多种语言模型和工具链。今天天气很好，想出去踏青。但是又比较懒不想出去，怎么办"

# 4.开始切割
texts = text_splitter.split_text(text)

print(f"原始文本被分割成了 {len(texts)} 个块:")
for i, chunk in enumerate(texts):
    print(f"块 {i+1}: 长度：{len(chunk)} 内容：{chunk}")
    print("-" * 50)

## 4、SemanticChunker：语义分块

举例：

In [ ]:
# pip install langchain_experimental
from langchain_community.embeddings import DashScopeEmbeddings
from langchain_experimental.text_splitter import SemanticChunker
import os
from dotenv import load_dotenv

load_dotenv(override=True)

# 加载文本
with open("../asset/load/09-ai1.txt", encoding="utf-8") as f:
    state_of_the_union = f.read()  #返回字符串

# 获取嵌入模型
embedding_model = DashScopeEmbeddings(
	model="qwen3.7-text-embedding",
	dashscope_api_key=os.getenv("DASHSCOPE_API_KEY"),
)


# 获取切割器
text_splitter = SemanticChunker(
    embeddings=embedding_model,   # 嵌入模型
    breakpoint_threshold_type="percentile", # 断点阈值类型：字面值["百分位数", "标准差", "四分位距", "梯度"] 选其一
    breakpoint_threshold_amount=65.0, # 断点阈值数量 (极低阈值 → 高分割敏感度)
    sentence_split_regex=r"(?<=[。？！])\s+" # 句子切分正则:遇到中文的句号、感叹号、问号（。？！）且后面带有空格时，先将其切分为独立的“句子”。
)

# 切分文档
docs = text_splitter.create_documents(texts = [state_of_the_union])

print(len(docs))
for doc in docs:
    print(f"🔍 文档: {doc}")

## 5、HTMLHeaderTextSplitter(了解)

举例

In [17]:
# 1.导入相关依赖
from langchain_text_splitters import HTMLHeaderTextSplitter

# 2.定义HTML文件
html_string = """
<!DOCTYPE html>
<html>
<body>
    <div>
        <h1>欢迎来到尚硅谷！</h1>
        <p>尚硅谷是专门培训IT技术方向</p>
        <div>
            <h2>尚硅谷老师简介</h2>
            <p>尚硅谷老师拥有多年教学经验，都是从一线互联网下来</p>
            <h3>尚硅谷北京校区</h3>
            <p>北京校区位于宏福科技园区</p>
        </div>
    </div>
</body>
</html>
"""

# 4.用于指定要根据哪些HTML标签来分割文本
headers_to_split_on = [
    ("h1", "标题1"),
    ("h2", "标题2"),
    ("h3", "标题3"),
]

# 5.定义HTMLHeaderTextSplitter分割器
html_splitter = HTMLHeaderTextSplitter(headers_to_split_on=headers_to_split_on)

# 6.分割器分割
html_header_splits = html_splitter.split_text(html_string)

print(html_header_splits)

[Document(metadata={'标题1': '欢迎来到尚硅谷！'}, page_content='欢迎来到尚硅谷！'), Document(metadata={'标题1': '欢迎来到尚硅谷！'}, page_content='尚硅谷是专门培训IT技术方向'), Document(metadata={'标题1': '欢迎来到尚硅谷！', '标题2': '尚硅谷老师简介'}, page_content='尚硅谷老师简介'), Document(metadata={'标题1': '欢迎来到尚硅谷！', '标题2': '尚硅谷老师简介'}, page_content='尚硅谷老师拥有多年教学经验，都是从一线互联网下来'), Document(metadata={'标题1': '欢迎来到尚硅谷！', '标题2': '尚硅谷老师简介', '标题3': '尚硅谷北京校区'}, page_content='尚硅谷北京校区'), Document(metadata={'标题1': '欢迎来到尚硅谷！', '标题2': '尚硅谷老师简介', '标题3': '尚硅谷北京校区'}, page_content='北京校区位于宏福科技园区')]


## 6、CodeTextSplitter(了解)

举例1：支持的语言

In [ ]:
from langchain_text_splitters import Language

# 支持分割语言类型
# Full list of supported languages
langs = [e.value for e in Language]
print(langs)

举例2：

In [ ]:
# 1.导入相关依赖
from langchain_text_splitters import Language, RecursiveCharacterTextSplitter
from pprint import pprint

# 2.定义要分割的python代码片段
PYTHON_CODE = """
def hello_world():
    print("Hello, World!")

def hello_world1():
    print("Hello, World1!")
"""

# 3.定义递归字符切分器
python_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON,
    chunk_size=50,
    chunk_overlap=0
)

# 4.文档切分
python_docs = python_splitter.create_documents(texts=[PYTHON_CODE])

pprint(python_docs)

## 7、MarkdownTextSplitter(了解)

举例：

In [18]:
from langchain_text_splitters import MarkdownTextSplitter

markdown_text = """
# 一级标题\n
这是一级标题下的内容\n\n
## 二级标题\n
- 二级下列表项1\n
- 二级下列表项2\n
"""

# 关键步骤：直接修改实例属性
splitter = MarkdownTextSplitter(chunk_size=30, chunk_overlap=0)
splitter._is_separator_regex = True  #  强制将分隔符视为正则表达式

# 执行分割
docs = splitter.create_documents(texts = [markdown_text])

# print(len(docs))

for i, doc in enumerate(docs):
    print(f"\n🔍 分块 {i + 1}:")
    print(doc.page_content)


🔍 分块 1:
# 一级标题

这是一级标题下的内容

🔍 分块 2:
## 二级标题

- 二级下列表项1

- 二级下列表项2
